# Introduction

Purpose: Develop a model that can detect small objects, such as birds, both stationary and in motion from a Rpi 5. 

Use Case: I have a RPi with a camera that is monitoring objects outside of my apartment unit. I would like to specifically detect small animals such as birds, squirrels and foxes. I do not care about detecting other classes such as people or people with dogs. 

Architecture:

- Apply transfer learning to a YOLO11 NCNN model that is optimized for RPi deployment
- Trade Study: SAHI to train model on different splits of the image
- Apply data augmentation
- Tiling during pre-processing and inference
- Nice to have: Omit classes


Data Acquisition
- Created my own dataset

Data Annotation
- already done by data set

Data Preprocessing
- resize if necessary
- normalize if necessary
- split
- cleaning
- tiling?

Data Augmentation
- Augment data in pipeline

Model Selection
- YOLOv11 NCNN
- use pre-trained weights...?

Training
- select appropriate loss function
- use adam optimizer
- Hyperparameters and tuning

Validation 
- learning rate schedulers and early stopping
- hyperparameter tuning
- asses validation loss accuracy and loss

Evaluation
- use test data to assess generalization....

Deploy
- ???

In [ ]:
pip install ultralytics ncnn

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from ultralytics import YOLO

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Data Augmentation and Training

In [ ]:
model = YOLO("yolo11n.pt", task='detect')
data_path = '/kaggle/input/yolo_bird_data/data.yaml'
# model.model # Only for manual parameter freezing or custom modificaitons (loss function)

In [ ]:
def train_yolo11_advanced(data_path, pretrained_model):
    """
    Advanced YOLO11 training with Adam optimizer, hyperparameter tuning,
    learning rate scheduling, and early stopping
    """
    model = pretrained_model
    
    # Training configuration
    train_config = {
        # Dataset
        'data': data_path,
        
        # Training parameters
        'epochs': 300,
        'batch': 8,  # Consider reducing if you have large/varying resolution images
        'imgsz': 832,
        'device': 0,  # GPU device, or 'cpu'
        
        # Optimizer settings
        'optimizer': 'Adam',  # Adam optimizer
        'lr0': 0.001,         # Initial learning rate
        'lrf': 0.01,          # Final learning rate (lr0 * lrf)
        'momentum': 0.937,    # Momentum (for SGD, but affects Adam's beta1)
        'weight_decay': 0.0005,  # Weight decay (L2 regularization)
        
        # Learning rate scheduler
        'cos_lr': True,       # Use cosine learning rate scheduler
        'warmup_epochs': 3,   # Warmup epochs
        'warmup_momentum': 0.8,  # Warmup momentum
        'warmup_bias_lr': 0.1,   # Warmup bias learning rate
        
        # Early stopping
        'patience': 50,       # Epochs to wait for improvement
        'save_period': 10,    # Save checkpoint every N epochs
        
        # Data augmentation - consider reducing for bird detection
        'hsv_h': 0.015,       # HSV-Hue augmentation
        'hsv_s': 0.5,         # HSV-Saturation augmentation
        'hsv_v': 0.4,         # HSV-Value augmentation
        'degrees': 5.0,       # Reduced rotation for birds (was 10.0)
        'translate': 0.1,     # Translation fraction
        'scale': 0.2,         # Scaling factor
        'shear': 0.5,         # Reduced shear for birds (was 1.0)
        'perspective': 0.0,   # Perspective transformation
        'flipud': 0.0,        # Vertical flip probability (birds don't fly upside down)
        'fliplr': 0.3,        # Reduced horizontal flip (was 0.5) - some birds are asymmetric
        'mosaic': 0.5,        # Reduced mosaic probability (was 0.75) for varying resolutions
        'mixup': 0.05,        # Reduced mixup (was 0.1) for cleaner bird features
        'copy_paste': 0.05,   # Reduced copy-paste (was 0.1)
        
        # Loss function weights (YOLO11 uses composite loss)
        'box': 7.5,           # Box loss weight
        'cls': 0.5,           # Classification loss weight
        'dfl': 1.5,           # Distribution focal loss weight
        
        # Training optimization - OPTIMIZED FOR 1920x1080 IMAGES
        'amp': True,          # Automatic Mixed Precision - crucial for large images
        'fraction': 1.0,      # Fraction of dataset to train on
        'profile': False,     # Profile training speed
        'freeze': None,       # Freeze layers (list of layer indices or None)
        'rect': True,         # ESSENTIAL for varying resolutions (1920x1080 vs 1080x1920)
        'cache': 'disk',      # Disk caching recommended for large images
        'multi_scale': True,  # ENABLE multi-scale training for varying resolutions
        
        # Validation
        'val': True,          # Validate during training
        'split': 'val',       # Dataset split to use for validation
        'save_json': False,   # Save results to JSON
        'save_hybrid': False, # Save hybrid version of labels
        'conf': 0.25,         # Confidence threshold for predictions
        'iou': 0.7,           # IoU threshold for NMS
        'max_det': 300,       # Maximum detections per image
        'half': False,        # Use half precision
        'dnn': False,         # Use OpenCV DNN backend
        'plots': True,        # Generate plots during training
        
        # Logging
        'verbose': True,      # Verbose output
        'seed': 0,            # Random seed for reproducibility
        'deterministic': True, # Deterministic training
        'single_cls': False,  # Single class training
        'resume': False,      # Resume training from last checkpoint
        'overlap_mask': True, # Overlap masks during training
        'mask_ratio': 4,      # Mask downsample ratio
        'dropout': 0.0,       # Dropout rate
    }
    
    # Train the model
    results = model.train(**train_config)
    
    return results

# Usage
results = train_yolo11_advanced(data_path, model)

# Validate, Test and Export for RPI5

In [ ]:
# Validate the model
metrics = model.val(data="/kaggle/input/bird-train-valid-test-split/data.yaml", split="val")  # no arguments needed, dataset and settings remembered
metrics.box.map  
metrics.box.map50  
metrics.box.map75  
metrics.box.maps  

# Test Model
metrics = model.val(data="/kaggle/input/bird-train-valid-test-split/data.yaml", split="test")  # no arguments needed, dataset and settings remembered
metrics.box.map  
metrics.box.map50  
metrics.box.map75  
metrics.box.maps

# Export model on Raspberry Pi, not on Kaggle VM
